# Data Preparation: Part 3 - Aircraft Type

## 1. Import Packages

In [1]:
import numpy as np 
import pandas as pd

from feature_engine.encoding import OneHotEncoder, RareLabelEncoder
from feature_engine.imputation import CategoricalImputer

import sklearn 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# from sklearn.pipeline import make_pipeline

## 2. Import Raw Data

In [ ]:
full = pd.read_csv("02_Data_Cleaning/Raw_Input_Files/asrs_full.csv.gz")

## 3. Data Cleaning & Augmentations

### 3.1. Prelim Data Cleaning

**NOTE**: There are several 'aircraft_type' observations with different cases, e.g. 
- "Commercial Fixed Wing" vs. "commercial fixed wing"
- "No Aircraft" vs. "No aircraft"
- "Small Aircraft" vs. "small aircraft"

In [3]:
# Replace 'aircraft_type' with all uppercase
data_cl = full
data_cl['aircraft_type'] = full['aircraft_type'].str.upper()
# data_cl.head()

**NOTE**: Fixing "UAV - Unpiloted Aerial Vehicle" vs. "UAV: Unpiloted Aerial Vehicle".

In [4]:
data_cl['aircraft_type'] = data_cl['aircraft_type'].replace("UAV: UNPILOTED AERIAL VEHICLE", "UAV - UNPILOTED AERIAL VEHICLE")

**NOTE**: Fixing "ANY UNKNOWN OR UNLISTED AIRCRAFT MANUFACTURER" to "UNKNOWN".

In [5]:
data_cl['aircraft_type'] = data_cl['aircraft_type'].replace("ANY UNKNOWN OR UNLISTED AIRCRAFT MANUFACTURER", "UNKNOWN")

### 3.2. Treat Missing Values

In [6]:
# Identify variables with any missing values
data_miss = data_cl.columns[full.isnull().any()].tolist()

In [7]:
# List missing categorical columns
# cat = data_raw.columns[data_raw.dtypes == 'object'].tolist()
cat = data_cl[data_miss].columns[data_cl[data_miss].dtypes == 'object'].tolist()
cat

['aircraft_type', 'reporter_function', 'flight_phase', 'synopsis']

In [8]:
# Replace categorical columns
imp = CategoricalImputer(imputation_method = 'missing', 
                         fill_value = 'UNKNOWN',
                         # variables = cat)
                         variables = 'aircraft_type')

data_cl = imp.fit_transform(data_cl)

### 3.3. Treat Rare Categories

In [9]:
data_cl[['aircraft_type']].value_counts()
# data_train[['aircraft_type']].value_counts()
# data_valid[['aircraft_type']].value_counts()

aircraft_type                       
COMMERCIAL FIXED WING                   4792
SKYHAWK 172/CUTLASS 172                 1516
B737-800                                1279
B737 UNDIFFERENTIATED OR OTHER MODEL    1272
A320                                     993
                                        ... 
AUTEL ROBOTICS EVO MAX 4N                  1
AUTEL ROBOTICS EVO                         1
SLING 2                                    1
SUPER HERCULES (C130J)                     1
ZODIAC CH601 / CH650                       1
Name: count, Length: 550, dtype: int64

In [10]:
# ?RareLabelEncoder

In [11]:
enc = RareLabelEncoder(tol=0.0015, 
                       variables = 'aircraft_type',
                       replace_with = 'OTHER')

data_enc = enc.fit_transform(data_cl)

In [12]:
data_enc[['aircraft_type']].value_counts()

aircraft_type                                     
COMMERCIAL FIXED WING                                 4792
OTHER                                                 4582
SKYHAWK 172/CUTLASS 172                               1516
B737-800                                              1279
B737 UNDIFFERENTIATED OR OTHER MODEL                  1272
                                                      ... 
BOEING COMPANY UNDIFFERENTIATED OR OTHER MODEL          54
EMB ERJ 135 ER/LR                                       53
AIRLINER 99                                             51
CESSNA CITATION SOVEREIGN (C680)                        51
REGIONAL JET CL65; UNDIFFERENTIATED OR OTHER MODEL      51
Name: count, Length: 91, dtype: int64

In [13]:
data_enc.head()

,acn,event_date,anomaly_code,location_id,aircraft_type,reporter_function,flight_phase,narrative_text,synopsis,word_count,hedge_score,passive_voice_ratio,causal_connective_count,specificity_score,named_entity_count,type_token_ratio,recurrence_flag,days_to_recurrence,is_right_censored,label_method
0,1507557,2018-01-01,ATC Issue All Types; Deviation / Discrepancy -...,C90.TRACON,A330,Approach,Landing,Aircraft X was assigned a runway approximately...,MLI Approach Controller reported ATC denied th...,143,0.000000,0.5556,2,0.00,7,0.5833,1,181.0,0,exact_anomaly_code_same_location_12mo
1,1513720,2018-01-01,Aircraft Equipment Problem Less Severe,ZZZ.Airport,EMB ERJ 170/175 ER/LR,Captain; Pilot Not Flying,Cruise,Pack 2 was MELed. My FO (First Officer) and I ...,ERJ-175 Captain reported diverting after exper...,318,0.003145,0.2105,3,0.75,19,0.4969,0,NaN,0,exact_anomaly_code_same_location_12mo
2,1513718,2018-01-01,Flight Deck / Cabin / Aircraft Event Illness /...,ZZZ.ARTCC,A321,Pilot Not Flying,Cruise,[A passenger] was reported as being ill and un...,A321 flight crew member reported difficulty re...,166,0.000000,0.1111,0,0.00,13,0.5689,1,120.0,0,exact_anomaly_code_same_location_12mo
3,1513706,2018-01-01,Aircraft Equipment Problem Critical,ZZZ.Airport,CITATION V/ULTRA/ENCORE (C560),First Officer; Pilot Flying,Final Approach,While on approach at 1;500ft; we asked the tow...,CE-560 First Officer reported that while on ap...,268,0.000000,0.0556,1,0.50,11,0.5221,0,NaN,0,exact_anomaly_code_same_location_12mo
4,1513663,2018-01-01,Aircraft Equipment Problem Less Severe; Deviat...,RCTP.Airport,B747-400,Pilot Flying; Captain,NaN,Our aircraft had just finished a 'heavy' maint...,B747-400 Captain reported a loss of hydraulic ...,190,0.000000,0.0667,0,0.25,5,0.5812,0,NaN,0,exact_anomaly_code_same_location_12mo


### 3.4. OneHotEncoder

In [14]:
data_enc2 = data_enc.copy()

In [15]:
# Clean / prepare 'aircraft_type' for variable names

rep = ["(", ")", ";"]
for x in rep:
    data_enc2['aircraft_type'] = data_enc2['aircraft_type'].str.replace(x, "", regex=False)

rep = [" - ", " / ", " ", "/", "-"]
for x in rep:
    data_enc2['aircraft_type'] = data_enc2['aircraft_type'].str.replace(x, "_", regex=False)

data_enc2['aircraft_type'] = data_enc2['aircraft_type'].str.replace("__", "_", regex=False)
data_enc2['aircraft_type'] = data_enc2['aircraft_type'].str.replace("__", "_", regex=False)

In [16]:
# View new 'aircraft_type' names

counts = data_enc2.value_counts(data_enc2['aircraft_type'])
with pd.option_context('display.max_rows', None):
    display(counts)

aircraft_type
COMMERCIAL_FIXED_WING                                  4792
OTHER                                                  4582
SKYHAWK_172_CUTLASS_172                                1516
B737_800                                               1279
B737_UNDIFFERENTIATED_OR_OTHER_MODEL                   1272
A320                                                    993
EMB_ERJ_170_175_ER_LR                                   871
SMALL_AIRCRAFT                                          871
A321                                                    814
EMB_ERJ_145_ER_LR                                       795
B737_700                                                784
A319                                                    784
PA_28_CHEROKEE_ARCHER_DAKOTA_PILLAN_WARRIOR             718
SMALL_AIRCRAFT_HIGH_WING_1_ENG_FIXED_GEAR               625
SMALL_AIRCRAFT_LOW_WING_1_ENG_FIXED_GEAR                563
REGIONAL_JET_900_CRJ900                                 528
REGIONAL_JET_200_ER_LR_CRJ

In [17]:
# ?OneHotEncoder

In [18]:
# Assign 33.7K 'aircraft_type' observations to a list
cat_list = data_enc2['aircraft_type'].tolist()

# Create list with unique 'aircraft_type' only
cat_list_unique = list(set(cat_list))

# Create a list with the 'aircraft_type' variable name and the unique list
categories = [('aircraft_type', cat_list_unique)]

In [19]:
# Set up OneHotEncoder
enc = OneHotEncoder(sparse_output=False, categories=[cat_list_unique])

# Fit
data_ohe = pd.DataFrame(
    enc.fit_transform(data_enc2[['aircraft_type']]),
    columns = enc.get_feature_names_out(),
    index = data_enc2.index)

In [20]:
# Concat original/initial-cleaned data with new dummies
data_ohe2 = pd.concat([data_cl.acn, data_enc2.aircraft_type, data_ohe], axis=1)

# If we want a version without 'aircraft_type':
# data_ohe2 = pd.concat([data_enc2.drop('aircraft_type', axis=1), data_ohe], axis=1)

# If we want a version with cleaned/rare encoded:
# data_ohe2 = pd.concat([data_enc2, data_ohe], axis=1)

In [21]:
len(full.aircraft_type.unique())

551

In [22]:
len(data_ohe2.aircraft_type.unique())

91

## 4. Export Data

In [ ]:
# Export
# data_ohe2.to_csv("02_Data_Cleaning/Output_Files/Data 3 - aircraft_type clean.csv", index = False)